# Implementation: Magnetism, Dynamo Action and the Solar-Stellar Connection
# 구현: 자기장, 다이나모 작용, 태양-항성 연결

**Paper 53**: Brun A. S., Browning M. K., *Living Reviews in Solar Physics*, **14**:4, 2017.
DOI: 10.1007/s41116-017-0007-8

This notebook reproduces six conceptual toy calculations from the review:
1. Rossby-activity relation (saturation + power-law)
2. Dynamo number $D = \alpha\Delta\Omega d^3/\eta^2$ across stellar parameter space
3. Kinematic $\alpha$-$\Omega$ dynamo wave (Parker-Yoshimura)
4. Babcock-Leighton toy model (tilted active region -> poleward flux -> reversal)
5. Equipartition field $B_{\rm eq}$ from photosphere to CZ base
6. HR-diagram schematic: fully convective, TO (tachocline), radiative-envelope regions

본 노트북은 리뷰의 여섯 가지 개념적 toy 계산을 재현:
1. Rossby-활동 관계 (포화 + 거듭제곱)
2. 별 매개변수 공간의 다이나모 수 $D$
3. 운동학적 $\alpha$-$\Omega$ 다이나모 파동 (Parker-Yoshimura)
4. Babcock-Leighton toy 모델 (기울어진 활동영역 -> 극방향 자속 -> 반전)
5. 광구에서 CZ 기저까지 등분배 장 $B_{\rm eq}$
6. HR 도표 개념도: 완전 대류, TO (터코클라인), 복사 외피 영역

## Setup / 설정

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 11
np.random.seed(42)

## 1. Rossby-Activity Relation / 로스비-활동 관계

Observations show X-ray activity saturates at $Ro < Ro_{\rm sat}\approx 0.13$ and declines as a power law $L_X/L_{\rm bol}\propto Ro^{-\beta}$ with $\beta\sim 2$ above.

관측 결과 X선 활동은 $Ro < 0.13$에서 포화되고, 그 이상에서 거듭제곱 $L_X/L_{\rm bol}\propto Ro^{-\beta}$ ($\beta\sim 2$) 감소.

In [ ]:
def activity_vs_rossby(Ro, Ro_sat=0.13, Lx_sat=1e-3, beta=2.0):
    """Compute L_X / L_bol as a function of Rossby number.

    Args:
        Ro: Rossby number (array or scalar).
        Ro_sat: Saturation Rossby number. Default 0.13.
        Lx_sat: Saturated L_X / L_bol ratio. Default 1e-3.
        beta: Power-law index in unsaturated regime. Default 2.0.

    Returns:
        L_X / L_bol evaluated at Ro.
    """
    Ro = np.asarray(Ro, dtype=float)
    return np.where(Ro < Ro_sat, Lx_sat, Lx_sat * (Ro / Ro_sat) ** (-beta))


Ro = np.logspace(-3, 1, 400)
lx = activity_vs_rossby(Ro)

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(Ro, lx, lw=2, color="C0")
ax.axvline(0.13, color="gray", ls="--", label=r"$Ro_{\rm sat}\approx 0.13$")
ax.axvline(2.0, color="orange", ls=":", label=r"Sun $Ro_\odot\approx 2$")
ax.set_xlabel("Rossby number $Ro$")
ax.set_ylabel(r"$L_X / L_{\rm bol}$")
ax.set_title("Rotation-activity relation / 회전-활동 관계")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Dynamo Number Across Stellar Parameter Space / 다이나모 수 매개변수 공간

$$D = \frac{\alpha\,\Delta\Omega\,d^{3}}{\eta^{2}}$$

We explore how $D$ varies with stellar mass (setting CZ depth $d$) and rotation period $P_{\rm rot}$ (setting $\Delta\Omega$ and helicity-proportional $\alpha$).

$D$가 항성 질량(CZ 깊이 $d$ 설정)과 자전 주기(차등회전 $\Delta\Omega$·헬리시티 비례 $\alpha$ 설정)에 따라 어떻게 변화하는지 탐색.

In [ ]:
def dynamo_number(mass_Msun, Prot_days, eta=1e12):
    """Compute dynamo number D for a main-sequence star.

    Uses crude mass-radius-CZ scalings to estimate alpha, delta-Omega, and d.

    Args:
        mass_Msun: Stellar mass in solar units.
        Prot_days: Rotation period in days.
        eta: Turbulent magnetic diffusivity in cgs. Default 1e12 cm^2/s.

    Returns:
        Dynamo number D (dimensionless).
    """
    R = 6.96e10 * (mass_Msun ** 0.9)
    if mass_Msun > 1.3:
        d = 0.05 * R
    elif mass_Msun > 0.35:
        d = (0.3 - 0.1 * (mass_Msun - 0.5)) * R
    else:
        d = R
    Omega = 2 * np.pi / (Prot_days * 86400.0)
    Teff = 5800 * (mass_Msun ** 0.5)
    dOmega = Omega * (Teff / 5800.0) ** 2 * 0.2
    vc = 50.0
    alpha = 0.3 * vc * (mass_Msun ** -0.5)
    D = alpha * dOmega * d ** 3 / eta ** 2
    return D


masses = np.linspace(0.2, 1.4, 60)
periods = np.logspace(np.log10(1.0), np.log10(60.0), 60)
M, P = np.meshgrid(masses, periods)
D = np.vectorize(dynamo_number)(M, P)

fig, ax = plt.subplots(figsize=(7.5, 5.5))
pcm = ax.pcolormesh(M, P, np.log10(np.abs(D) + 1e-3), cmap="viridis", shading="auto")
cs = ax.contour(M, P, np.log10(np.abs(D) + 1e-3), levels=[1, 2, 3, 4], colors="white")
ax.clabel(cs, inline=True, fmt=lambda v: rf"$\log|D|={int(v)}$")
ax.axhline(25, color="orange", ls=":", label="Sun (25 d)")
ax.axvline(1.0, color="orange", ls=":")
ax.set_yscale("log")
ax.set_xlabel("Mass $M/M_\\odot$")
ax.set_ylabel("Rotation period $P_{\\rm rot}$ [days]")
ax.set_title("Dynamo number across parameter space / 다이나모 수")
cbar = plt.colorbar(pcm, ax=ax)
cbar.set_label(r"$\log_{10}|D|$")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

print(f"Sun (1 Msun, 25 d): D = {dynamo_number(1.0, 25.0):.1f}")
print(f"Young Sun (1 Msun, 5 d): D = {dynamo_number(1.0, 5.0):.1f}")
print(f"M-dwarf (0.3 Msun, 10 d): D = {dynamo_number(0.3, 10.0):.1f}")

## 3. Kinematic α-Ω Dynamo Wave / 운동학적 α-Ω 다이나모 파동

We integrate a simplified 1-D mean-field α-Ω system in latitude $\theta$:

$$\partial_t A = \alpha(\theta)\, B + \eta\,\partial_\theta^2 A$$
$$\partial_t B = \Omega'(\theta)\,\partial_\theta A + \eta\,\partial_\theta^2 B$$

The Parker-Yoshimura sign rule: waves propagate where $s = \alpha \nabla\Omega \times \hat e_\phi$; with $\alpha > 0$ in N and $\partial\Omega/\partial r < 0$, the wave propagates equatorward.

Parker-Yoshimura 부호규칙: 파동은 $s = \alpha \nabla\Omega \times \hat e_\phi$ 방향 전파; 북반구 $\alpha > 0$, $\partial\Omega/\partial r < 0$이면 적도 방향 전파.

In [ ]:
def alpha_omega_wave(N_theta=256, T=400, dt=0.01, alpha0=1.0, Omega_prime=-5.0, eta=1e-3):
    """Integrate 1-D alpha-Omega dynamo wave via explicit finite differences.

    Args:
        N_theta: Number of latitude grid points (-pi/2 to pi/2).
        T: Total integration time (dimensionless).
        dt: Timestep.
        alpha0: Peak alpha-effect amplitude.
        Omega_prime: Radial shear dOmega/dr (sign sets wave direction).
        eta: Turbulent diffusivity.

    Returns:
        (theta, times, B_history) arrays for plotting butterfly diagram.
    """
    theta = np.linspace(-np.pi / 2, np.pi / 2, N_theta)
    dtheta = theta[1] - theta[0]
    A = 0.01 * np.sin(2 * theta)
    B = np.zeros_like(A)
    alpha = alpha0 * np.sin(theta) * np.cos(theta)
    times = np.arange(0, T, dt)
    snapshots = max(1, len(times) // 300)
    B_history = []
    t_record = []

    for i, t in enumerate(times):
        A_xx = np.zeros_like(A)
        B_xx = np.zeros_like(B)
        A_xx[1:-1] = (A[2:] - 2 * A[1:-1] + A[:-2]) / dtheta ** 2
        B_xx[1:-1] = (B[2:] - 2 * B[1:-1] + B[:-2]) / dtheta ** 2
        A_theta = np.zeros_like(A)
        A_theta[1:-1] = (A[2:] - A[:-2]) / (2 * dtheta)
        dAdt = alpha * B + eta * A_xx
        dBdt = Omega_prime * A_theta + eta * B_xx
        A = A + dt * dAdt
        B = B + dt * dBdt
        if i % snapshots == 0:
            B_history.append(B.copy())
            t_record.append(t)
    return theta, np.array(t_record), np.array(B_history)


theta, t_rec, B_hist = alpha_omega_wave()
lat = np.degrees(theta)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
vmax = np.abs(B_hist).max()
pcm = ax.pcolormesh(t_rec, lat, B_hist.T, cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto")
ax.set_xlabel("Time")
ax.set_ylabel("Latitude [deg]")
ax.set_title("Toroidal field $B_\\phi(\\theta, t)$ / 토로이달 장")
plt.colorbar(pcm, ax=ax, label=r"$B_\phi$")

ax = axes[1]
for idx in [len(B_hist) // 4, len(B_hist) // 2, 3 * len(B_hist) // 4, len(B_hist) - 1]:
    ax.plot(lat, B_hist[idx], label=f"t = {t_rec[idx]:.0f}")
ax.set_xlabel("Latitude [deg]")
ax.set_ylabel(r"$B_\phi$")
ax.set_title("Snapshots / 스냅샷")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Babcock-Leighton Toy Model / 배브콕-레이턴 toy 모델

At the surface, tilted bipolar active regions (Joy's law) contribute net poloidal field via their leading/following spot latitudinal separation. This "surface source" poleward-migrates via meridional flow and reverses polar cap polarity at cycle peak.

표면에서 기울어진 쌍극 활동영역(Joy 법칙)은 선도/후행 흑점의 위도 분리로 순 폴로이달 장을 기여. 이 "표면 원천"은 자오면 유동으로 극 방향 이동하여 주기 극대에서 극 덮개 극성 반전.

In [ ]:
def babcock_leighton_surface(t_years, n_regions=20, tilt_deg=5.0, cycle_yr=11.0):
    """Simulate surface poloidal flux from tilted ARs migrating poleward.

    Args:
        t_years: Time array in years.
        n_regions: Number of AR emergence events per cycle.
        tilt_deg: Active region tilt angle in degrees.
        cycle_yr: Cycle period in years.

    Returns:
        (lat_grid, flux) — flux as (time, latitude) array.
    """
    lat_grid = np.linspace(-90, 90, 181)
    flux = np.zeros((len(t_years), len(lat_grid)))
    v_merid = 5.0  # deg/yr poleward drift
    for k in range(int(t_years[-1] / cycle_yr) + 2):
        t_cycle_start = k * cycle_yr
        polarity = (-1) ** k
        for j in range(n_regions):
            t_emerge = t_cycle_start + (j / n_regions) * cycle_yr
            phase = (t_emerge - t_cycle_start) / cycle_yr
            lat_mean = 30 - 25 * phase
            for hemi_sign in [+1, -1]:
                center = hemi_sign * lat_mean
                local_tilt = np.sign(center) * tilt_deg
                for dlat, sign in [(-local_tilt, +1), (+local_tilt, -1)]:
                    pos0 = center + dlat
                    for i_t, tt in enumerate(t_years):
                        if tt >= t_emerge:
                            dt_since = tt - t_emerge
                            shifted = pos0 + hemi_sign * v_merid * dt_since
                            flux[i_t] += (sign * polarity * hemi_sign *
                                          np.exp(-(lat_grid - shifted) ** 2 / 10 ** 2) *
                                          np.exp(-dt_since / cycle_yr))
    return lat_grid, flux


t_yr = np.linspace(0, 33, 300)
lat_grid, flux = babcock_leighton_surface(t_yr)

pol_N = flux[:, lat_grid > 60].mean(axis=1)
pol_S = flux[:, lat_grid < -60].mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
vmax = np.abs(flux).max() * 0.5 or 1.0
pcm = ax.pcolormesh(t_yr, lat_grid, flux.T, cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto")
ax.set_xlabel("Time [yr]")
ax.set_ylabel("Latitude [deg]")
ax.set_title("Surface radial flux (BL toy) / BL 표면 자속")
plt.colorbar(pcm, ax=ax)

ax = axes[1]
ax.plot(t_yr, pol_N, label="N polar", color="C3")
ax.plot(t_yr, pol_S, label="S polar", color="C0")
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("Time [yr]")
ax.set_ylabel("Mean polar flux")
ax.set_title("Polar cap polarity reversal / 극 덮개 극성 반전")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Equipartition Field from Photosphere to CZ Base / 광구에서 CZ 기저까지 등분배 장

$$B_{\rm eq}(r) = \sqrt{4\pi\rho(r)}\,v_{\rm turb}(r)$$

Using MLT, $v_{\rm turb}\propto (F/\rho)^{1/3}$, so $B_{\rm eq}^2\propto \rho^{1/3}F^{2/3}$.

MLT로 $v_{\rm turb}\propto (F/\rho)^{1/3}$이므로 $B_{\rm eq}^2\propto \rho^{1/3}F^{2/3}$.

In [ ]:
def equipartition_profile(r_over_R, rho_phot=3e-7, rho_base=0.21, flux_norm=6.3e10):
    """Compute B_eq(r) from photosphere (r=1) to CZ base (r=0.72) for the Sun.

    Args:
        r_over_R: Normalized radius array.
        rho_phot: Photospheric density in cgs.
        rho_base: Density at CZ base in cgs.
        flux_norm: Solar radiative flux (erg/cm^2/s).

    Returns:
        (rho, v_turb, B_eq) profiles.
    """
    rho = rho_phot * (rho_base / rho_phot) ** ((1 - r_over_R) / (1 - 0.72))
    v_turb = (flux_norm / rho) ** (1.0 / 3.0) * 1e-3
    B_eq = np.sqrt(4 * np.pi * rho) * v_turb
    return rho, v_turb, B_eq


r = np.linspace(0.72, 1.0, 200)
rho, v_turb, B_eq = equipartition_profile(r)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

ax = axes[0]
ax.semilogy(r, rho, color="C0", lw=2)
ax.set_xlabel(r"$r/R_\odot$")
ax.set_ylabel(r"$\rho$ [g/cm$^3$]")
ax.set_title("Density / 밀도")
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(r, v_turb, color="C1", lw=2)
ax.set_xlabel(r"$r/R_\odot$")
ax.set_ylabel(r"$v_{\rm turb}$ [arb]")
ax.set_title("Turbulent velocity (MLT) / 난류 속도")
ax.grid(alpha=0.3)

ax = axes[2]
ax.semilogy(r, B_eq, color="C3", lw=2)
ax.axhline(400, color="gray", ls="--", label="~400 G (photosphere)")
ax.axhline(1e4, color="gray", ls=":", label=r"~$10^4$ G (base)")
ax.set_xlabel(r"$r/R_\odot$")
ax.set_ylabel(r"$B_{\rm eq}$ [G]")
ax.set_title("Equipartition field / 등분배 장")
ax.legend()
ax.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

print(f"B_eq at photosphere (r=1):    {B_eq[-1]:.1f} G")
print(f"B_eq at CZ base (r=0.72):     {B_eq[0]:.1f} G")

## 6. HR Diagram Schematic of Convection Regimes / HR 도표 대류 영역 개념도

- $M < 0.35\,M_\odot$: **Fully convective** (no radiative core).
- $0.35 \leq M \leq 1.8\,M_\odot$: **Solar-like** — radiative interior + convective envelope + **tachocline** (TO) between them.
- $M > 1.8\,M_\odot$: **Convective core + radiative envelope**.

-   $M < 0.35\,M_\odot$: **완전 대류**.
-   $0.35 \leq M \leq 1.8\,M_\odot$: **태양형** — 복사 중심 + 대류 외피 + **터코클라인(TO)**.
-   $M > 1.8\,M_\odot$: **대류 중심 + 복사 외피**.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

Teff_grid = np.logspace(np.log10(2500), np.log10(40000), 400)
L_MS = (Teff_grid / 5800.0) ** 5
ax.plot(Teff_grid, L_MS, "k-", alpha=0.6, lw=2, label="Main sequence")

mass_to_Teff = lambda M: 5800 * M ** 0.5
T_035 = mass_to_Teff(0.35)
T_18 = mass_to_Teff(1.8)

ax.axvspan(2500, T_035, alpha=0.25, color="red", label=r"Fully convective ($M < 0.35\,M_\odot$)")
ax.axvspan(T_035, T_18, alpha=0.25, color="gold", label="Convective envelope + tachocline")
ax.axvspan(T_18, 40000, alpha=0.25, color="lightblue", label="Convective core + radiative envelope")

ax.plot([5800], [1.0], "o", color="orange", markersize=14, markeredgecolor="k", label="Sun", zorder=5)
ax.plot([3050], [1.5e-3], "o", color="red", markersize=9, markeredgecolor="k", label="Proxima Cen", zorder=5)
ax.plot([5790], [1.52], "o", color="yellow", markersize=10, markeredgecolor="k", label=r"$\alpha$ Cen A", zorder=5)
ax.plot([9600], [40], "o", color="white", markersize=11, markeredgecolor="k", label="Vega (A0V)", zorder=5)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(40000, 2500)
ax.set_ylim(1e-4, 1e5)
ax.set_xlabel(r"$T_{\rm eff}$ [K]")
ax.set_ylabel(r"$L / L_\odot$")
ax.set_title("HR diagram: stellar convection regimes / 항성 대류 영역")
ax.legend(loc="lower left", fontsize=9)
ax.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

## Summary / 요약

### English
- **Rossby-activity saturation** is reproduced as a broken power law with $Ro_{\rm sat}\approx 0.13$; the Sun sits well above saturation at $Ro\approx 2$.
- **Dynamo number** varies by many orders of magnitude across the HR diagram; the Sun is mildly supercritical ($D\sim 10^2$), consistent with a weakly excited cycle.
- The **α-Ω wave** integration demonstrates Parker-Yoshimura propagation: sign of $\alpha\partial\Omega/\partial r$ sets equatorward or poleward migration.
- The **Babcock-Leighton toy** recovers polar-flux reversal on cycle timescale via AR tilt + meridional advection.
- The **equipartition profile** spans $\sim 400$ G at the photosphere to $\sim 10^4$ G near the CZ base—far below the $10^5$ G required for classical flux emergence, highlighting the need for concentration at the tachocline.
- The **HR-diagram regimes** make concrete how stellar structure (and thus dynamo type) depends on mass.

### 한국어
- **Rossby-활동 포화**는 $Ro_{\rm sat}\approx 0.13$의 꺾인 거듭제곱으로 재현; 태양은 포화 위 $Ro\approx 2$에 위치.
- **다이나모 수**는 HR 도표 전반에 걸쳐 여러 차수 변동; 태양은 약간 초임계($D\sim 10^2$)로 약하게 여기된 주기와 일치.
- **α-Ω 파동** 적분은 Parker-Yoshimura 전파를 입증: $\alpha\partial\Omega/\partial r$ 부호가 적도/극 방향 이동 결정.
- **Babcock-Leighton toy**는 AR 기울기 + 자오면 이류를 통해 주기 시간규모의 극 자속 반전을 재현.
- **등분배 프로파일**은 광구 $\sim 400$ G에서 CZ 기저 $\sim 10^4$ G—고전 자속 분출 필요치 $10^5$ G보다 훨씬 낮아 터코클라인 집중 필요성 부각.
- **HR 도표 영역**은 항성 구조(및 다이나모 유형)가 질량에 따라 어떻게 달라지는지를 구체화.